# aiohttp_cors: 5 Basic Usage Examples

In [ ]:
# Example 1: Basic aiohttp server with CORS enabled for all routes
import aiohttp
import aiohttp.web
import aiohttp_cors

app = aiohttp.web.Application()
async def hello(request):
    return aiohttp.web.Response(text="Hello, world!")
app.router.add_get("/", hello)
cors = aiohttp_cors.setup(app, defaults={
    '*': aiohttp_cors.ResourceOptions(
        allow_credentials=True,
        expose_headers='*',
        allow_headers='*',
    )
})
for route in list(app.router.routes()):
    cors.add(route)
runner = aiohttp.web.AppRunner(app)
await runner.setup()
site = aiohttp.web.TCPSite(runner, '127.0.0.1', 0)  # port 0 = any free port
await site.start()
try:
    bound_port = runner.addresses[0][1]
    print(f'Server started at http://127.0.0.1:{bound_port}/')
    async with aiohttp.ClientSession() as session:
        async with session.get(f'http://127.0.0.1:{bound_port}/', headers={'Origin': 'http://example.com'}) as resp:
            body = await resp.text()
            allow_origin = resp.headers.get('Access-Control-Allow-Origin')
    assert body == "Hello, world!", body
    assert allow_origin == 'http://example.com', allow_origin
    print(f'Access-Control-Allow-Origin: {allow_origin}')
finally:
    await runner.cleanup()


In [ ]:
# Example 2: Restrict CORS to specific origins
import aiohttp
import aiohttp.web
import aiohttp_cors

app = aiohttp.web.Application()
async def hello(request):
    return aiohttp.web.Response(text="CORS restricted!")
app.router.add_get("/restricted", hello)
cors = aiohttp_cors.setup(app, defaults={
    'http://example.com': aiohttp_cors.ResourceOptions()
})
for route in list(app.router.routes()):
    cors.add(route)
runner = aiohttp.web.AppRunner(app)
await runner.setup()
site = aiohttp.web.TCPSite(runner, '127.0.0.1', 0)
await site.start()
try:
    url = f'http://127.0.0.1:{runner.addresses[0][1]}/restricted'
    print(f'Restricted CORS server at {url}')
    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers={'Origin': 'http://example.com'}) as resp:
            allowed = resp.headers.get('Access-Control-Allow-Origin')
        async with session.get(url, headers={'Origin': 'http://other.example'}) as resp:
            other_status = resp.status  # the request is still served; only the CORS header is withheld
            refused = resp.headers.get('Access-Control-Allow-Origin')
    assert allowed == 'http://example.com', allowed
    assert other_status == 200 and refused is None, (other_status, refused)
    print(f'http://example.com -> {allowed}; http://other.example -> no Access-Control-Allow-Origin')
finally:
    await runner.cleanup()


In [ ]:
# Example 3: CORS with custom allowed headers and methods
import aiohttp
import aiohttp.web
import aiohttp_cors

app = aiohttp.web.Application()
async def custom(request):
    return aiohttp.web.Response(text="Custom CORS headers and methods!")
route = app.router.add_route('POST', '/custom', custom)
cors = aiohttp_cors.setup(app, defaults={
    '*': aiohttp_cors.ResourceOptions(
        allow_methods=['POST'],
        allow_headers=['X-Custom-Header'],
    )
})
cors.add(route)
runner = aiohttp.web.AppRunner(app)
await runner.setup()
site = aiohttp.web.TCPSite(runner, '127.0.0.1', 0)
await site.start()
try:
    url = f'http://127.0.0.1:{runner.addresses[0][1]}/custom'
    print(f'Custom CORS server at {url}')
    preflight = {
        'Origin': 'http://example.com',
        'Access-Control-Request-Method': 'POST',
        'Access-Control-Request-Headers': 'X-Custom-Header',
    }
    async with aiohttp.ClientSession() as session:
        async with session.options(url, headers=preflight) as resp:
            status = resp.status
            methods = resp.headers.get('Access-Control-Allow-Methods', '')
            headers = resp.headers.get('Access-Control-Allow-Headers', '')
    assert status == 200, status
    assert 'POST' in methods and 'x-custom-header' in headers.lower(), (methods, headers)
    print(f'preflight allows methods={methods!r} headers={headers!r}')
finally:
    await runner.cleanup()


In [ ]:
# Example 4: CORS with credentials support
import aiohttp
import aiohttp.web
import aiohttp_cors

app = aiohttp.web.Application()
async def credentials(request):
    return aiohttp.web.Response(text="CORS with credentials!")
route = app.router.add_get('/credentials', credentials)
cors = aiohttp_cors.setup(app, defaults={
    '*': aiohttp_cors.ResourceOptions(
        allow_credentials=True
    )
})
cors.add(route)
runner = aiohttp.web.AppRunner(app)
await runner.setup()
site = aiohttp.web.TCPSite(runner, '127.0.0.1', 0)
await site.start()
try:
    url = f'http://127.0.0.1:{runner.addresses[0][1]}/credentials'
    print(f'CORS with credentials at {url}')
    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers={'Origin': 'http://example.com'}) as resp:
            body = await resp.text()
            allow_origin = resp.headers.get('Access-Control-Allow-Origin')
            allow_credentials = resp.headers.get('Access-Control-Allow-Credentials')
    assert body == "CORS with credentials!", body
    assert allow_origin == 'http://example.com' and allow_credentials == 'true', (allow_origin, allow_credentials)
    print(f'Access-Control-Allow-Origin={allow_origin} Access-Control-Allow-Credentials={allow_credentials}')
finally:
    await runner.cleanup()


In [ ]:
# Example 5: preflight OPTIONS is handled automatically for a GET-only route
import aiohttp
import aiohttp.web
import aiohttp_cors

app = aiohttp.web.Application()

async def handle_request(request):
    return aiohttp.web.Response(text="CORS request handled!")

# Only add GET route, let CORS handle OPTIONS automatically
app.router.add_route('GET', '/api', handle_request)

cors = aiohttp_cors.setup(app, defaults={
    '*': aiohttp_cors.ResourceOptions(
        allow_methods=['GET', 'OPTIONS'],
        allow_headers='*',
    )
})

# Add CORS to all routes
for route in list(app.router.routes()):
    cors.add(route)

runner = aiohttp.web.AppRunner(app)
await runner.setup()
site = aiohttp.web.TCPSite(runner, '127.0.0.1', 0)
await site.start()
try:
    url = f'http://127.0.0.1:{runner.addresses[0][1]}/api'
    print(f'CORS server with a GET-only route at {url}')
    preflight = {'Origin': 'http://example.com', 'Access-Control-Request-Method': 'GET'}
    async with aiohttp.ClientSession() as session:
        async with session.options(url, headers=preflight) as resp:
            status = resp.status
            methods = resp.headers.get('Access-Control-Allow-Methods', '')
        async with session.get(url, headers={'Origin': 'http://example.com'}) as resp:
            body = await resp.text()
            allow_origin = resp.headers.get('Access-Control-Allow-Origin')
    assert status == 200, status
    assert 'GET' in methods, methods
    assert body == "CORS request handled!" and allow_origin == 'http://example.com', (body, allow_origin)
    print(f'preflight OPTIONS -> {status}, Access-Control-Allow-Methods={methods!r}')
finally:
    await runner.cleanup()
